## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **24c7c78e-a1b4-4fe9-9936-6ad6aa5f636e**


## Dataset

# **Wasserverbrauch im Kanton Basel-Stadt und in Binningen seit 1951**


## Description

Der Datensatz „Wasserverbrauch im Kanton Basel-Stadt und in Binningen“ enthält Informationen zum Wasserverbrauch in dieser Region. Er umfasst jährliche Verbrauchsdaten seit 1951 sowie monatliche Verbrauchsdaten seit 2005. Die Daten stammen von den Industriellen Werken Basel &#40;IWB&#41; und werden vom Statistischen Amt des Kantons Basel-Stadt veröffentlicht. Die Erhebung erfolgt durch öffentliche Organe, insbesondere die IWB, wobei die Daten jeweils zum Monats- und Jahresende erfasst werden. Der Datensatz enthält verschiedene Kategorien des Wasserverbrauchs, darunter Haushaltungen und Gewerbe, Grossbezüger wie Industriebetriebe, öffentliche Brunnen sowie weitere öffentliche Zwecke. Zudem sind der Eigenbedarf der IWB und die Wasserverluste im Versorgungsnetz erfasst. Neben diesen Gesamtverbräuchen gibt es auch Angaben zum mittleren und grössten Tagesverbrauch pro Kopf. Die Daten werden in Litern angegeben und auf Jahres- oder Monatsebene aggregiert. Sie basieren auf der Wasserabgabe im eigenen Versorgungsgebiet, das den Kanton Basel-Stadt und Binningen umfasst, ohne Berücksichtigung von Transitlieferungen an Gemeinden ausserhalb des Versorgungsgebiets. Bis 1985 wurden Haushaltungen ohne Gewerbe separat ausgewiesen, während Industrie und Gewerbe bis zu diesem Zeitpunkt gemeinsam erfasst wurden. Die Kategorie „Öffentliche Brunnen“ umfasst Brunnen im Stadtgebiet, die durch die IWB betrieben und unterhalten werden. Zudem wurden die Werte für die Jahre 2015 bis 2019 im Jahr 2021 revidiert.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/wasserverbrauch-im-kanton-basel-stadt-und-in-binningen-seit-1951)


## Metadata

- **identifier:** 100420@kanton-basel-stadt
- **publisher.name:** Statistisches Amt
- **organization.display&#95;name:** Kanton Basel-Stadt
- **organization.url:** https://data.bs.ch
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** wasser
- **issued:** 2025-02-17 00:00:00
- **metadata&#95;created:** 2025-02-18T01:40:45.420756
- **metadata&#95;modified:** 2026-09-15T00:45:29.797517
- **Resource id:** 1d05ba29-a12c-4710-a3ca-0734498e9304
- **Resource name:** csv
- **Resource package&#95;id:** 24c7c78e-a1b4-4fe9-9936-6ad6aa5f636e
- **Resource description:** Wasserverbrauch im Kanton Basel-Stadt und in Binningen seit 1951 &#40;csv&#41;
- **Resource issued:** 2025-02-17 00:00:00
- **Resource modified:** 2025-08-04 08:45:59.100000+00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;by
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;by


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **csv** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://data.bs.ch/api/v2/catalog/datasets/100420/exports/csv?use_labels=true")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** opendata@bs.ch &#124; Open Data Basel-Stadt
